# Week 4 Reference: Circular Queues, Linked Lists & O(1) LRU Cache

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Companion runnable reference implementations for Lab 3.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-04/labs/W04_Reference_Queues_Linked_Lists.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Circular Array Queue with Amortized Wrap-Around Resize

A circular queue tracks `_front` and `_size`, computing the rear index via `(front + size) % capacity`. When full, unrolling items in logical order into a $2\times$ buffer preserves FIFO order.

In [1]:
class CircularArrayQueue:
    def __init__(self, cap=4):
        self._data = [None] * cap
        self._front = 0
        self._size = 0

    def __len__(self):
        return self._size

    def _resize(self, new_cap):
        block = [None] * new_cap
        for k in range(self._size):
            block[k] = self._data[(self._front + k) % len(self._data)]
        self._data = block
        self._front = 0

    def enqueue(self, x):
        if self._size == len(self._data):
            self._resize(2 * len(self._data))
        self._data[(self._front + self._size) % len(self._data)] = x
        self._size += 1

    def dequeue(self):
        if self._size == 0:
            raise IndexError("empty")
        v = self._data[self._front]
        self._data[self._front] = None
        self._front = (self._front + 1) % len(self._data)
        self._size -= 1
        return v


q = CircularArrayQueue(3)
for i in range(1, 7):
    q.enqueue(i)
out = [q.dequeue() for _ in range(4)]
q.enqueue(99)
print("Dequeued:", out, "| remaining size:", len(q))
assert out == [1, 2, 3, 4]


Dequeued: [1, 2, 3, 4] | remaining size: 3


## 2. Sentinel Doubly Linked List & $O(1)$ LRU Cache

Dummy `head` and `tail` sentinel nodes eliminate boundary `None` checks on insertion and removal. Pairing a hash map with a sentinel doubly linked list yields $O(1)$ `get` and `put` operations for an LRU cache.

In [2]:
class DNode:
    def __init__(self, key=None, val=None):
        self.key, self.val = key, val
        self.prev = self.next = None


class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _push_front(self, node):
        nxt = self.head.next
        node.prev, node.next = self.head, nxt
        self.head.next = nxt.prev = node

    def get(self, key):
        if key not in self.map:
            return None
        node = self.map[key]
        self._unlink(node)
        self._push_front(node)
        return node.val

    def put(self, key, val):
        if key in self.map:
            node = self.map[key]
            node.val = val
            self._unlink(node)
            self._push_front(node)
            return
        if len(self.map) == self.cap:
            lru = self.tail.prev
            self._unlink(lru)
            del self.map[lru.key]
        node = DNode(key, val)
        self.map[key] = node
        self._push_front(node)


lru = LRUCache(2)
lru.put("Pemba", 142)
lru.put("Unguja", 358)
assert lru.get("Pemba") == 142
lru.put("Mafia", 75)
assert lru.get("Unguja") is None
print("LRU Cache eviction verified: Pemba =", lru.get("Pemba"), "| Mafia =", lru.get("Mafia"))


LRU Cache eviction verified: Pemba = 142 | Mafia = 75
